<a href="https://colab.research.google.com/github/dhruvjoshi0905/Hack-O-Week/blob/main/hacko_o_week(12).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cryptography pandas

In [2]:
"""
Project: Wearable Data Ingestion & Encryption Pipeline (Logic Simulation)
Module Focus: Data Generation, Cryptography, and Storage Logic (Backend routing handled separately)
"""

import json
import time
import random
import pandas as pd
from datetime import datetime
from cryptography.fernet import Fernet

# ==========================================
# 1. INITIALIZATION & SECURITY SETUP
# ==========================================
# In a real app, this key would be hidden in an .env file.
# Fernet is an implementation of AES (Advanced Encryption Standard).
print("--- Initializing Security ---")
encryption_key = Fernet.generate_key()
cipher_suite = Fernet(encryption_key)
print(f"Generated AES Key: {encryption_key.decode('utf-8')}\n")

# This list will act as our simulated database for this Colab environment
mock_database = []

# ==========================================
# 2. THE DATASET: WEARABLE SIMULATOR
# ==========================================
def generate_wearable_data(device_id="watch_A1"):
    """Simulates a wearable device capturing heart rate and step count."""
    data = {
        "device_id": device_id,
        "timestamp": datetime.now().isoformat(),
        "heart_rate_bpm": random.randint(60, 120),  # Normal resting/active heart rate
        "steps_interval": random.randint(0, 50)     # Steps taken in the last few seconds
    }
    return data

# ==========================================
# 3. INGESTION & ENCRYPTION PIPELINE
# ==========================================
def ingest_data(raw_data):
    """
    Simulates the API receiving data, encrypting it, and storing it.
    This protects sensitive biometric data at rest.
    """
    # Convert the Python dictionary to a JSON string, then encode it to bytes
    json_string = json.dumps(raw_data)
    byte_data = json_string.encode('utf-8')

    # Encrypt the byte data
    encrypted_payload = cipher_suite.encrypt(byte_data)

    # "Insert" into our mock database
    db_record = {
        "id": len(mock_database) + 1,
        "encrypted_blob": encrypted_payload,
        "stored_at": datetime.now().strftime("%H:%M:%S")
    }
    mock_database.append(db_record)

    print(f"Ingested & Encrypted Record #{db_record['id']}")

# ==========================================
# 4. EXECUTE THE SIMULATION
# ==========================================
print("--- Starting Real-Time Data Ingestion ---")
# Simulate the watch sending 5 bursts of data, one second apart
for _ in range(5):
    live_data = generate_wearable_data()
    print(f"Device Transmitted: {live_data['heart_rate_bpm']} BPM")
    ingest_data(live_data)
    time.sleep(1) # Wait 1 second before next transmission

# ==========================================
# 5. VERIFICATION (PROVING IT WORKS)
# ==========================================
print("\n--- Verifying Database Contents ---")
# Show what the database actually looks like (it should be unreadable)
db_df = pd.DataFrame(mock_database)
print("What the database sees (Encrypted):")
print(db_df[['id', 'encrypted_blob']].head(2))

print("\n--- Decryption Test ---")
# Prove that we can retrieve and decrypt the data when an authorized user requests it
for record in mock_database:
    decrypted_bytes = cipher_suite.decrypt(record["encrypted_blob"])
    decrypted_dict = json.loads(decrypted_bytes.decode('utf-8'))

    print(f"Record #{record['id']} Decrypted: {decrypted_dict['heart_rate_bpm']} BPM at {decrypted_dict['timestamp']}")

--- Initializing Security ---
Generated AES Key: UilVl7ZxMuQP6-2h6_jURrVt4lxFT--ro61f7N74Fxo=

--- Starting Real-Time Data Ingestion ---
Device Transmitted: 75 BPM
Ingested & Encrypted Record #1
Device Transmitted: 86 BPM
Ingested & Encrypted Record #2
Device Transmitted: 68 BPM
Ingested & Encrypted Record #3
Device Transmitted: 109 BPM
Ingested & Encrypted Record #4
Device Transmitted: 95 BPM
Ingested & Encrypted Record #5

--- Verifying Database Contents ---
What the database sees (Encrypted):
   id                                     encrypted_blob
0   1  b'gAAAAABpzVcbl3v7fNlJ0hUcahtatqCbzcAp3FYltqiK...
1   2  b'gAAAAABpzVccwyFd-tTckSMqq2xiaO0So-VBnYa8prfI...

--- Decryption Test ---
Record #1 Decrypted: 75 BPM at 2026-04-01T17:34:19.835254
Record #2 Decrypted: 86 BPM at 2026-04-01T17:34:20.840926
Record #3 Decrypted: 68 BPM at 2026-04-01T17:34:21.841545
Record #4 Decrypted: 109 BPM at 2026-04-01T17:34:22.843475
Record #5 Decrypted: 95 BPM at 2026-04-01T17:34:23.844061
